# Matching System Prototype

Questo notebook implementa un primo prototipo funzionante del sistema di matching cane–famiglia.

L'obiettivo è usare:

- le caratteristiche strutturate dei cani;
- le preferenze espresse dalla famiglia;
- la descrizione testuale del cane ideale;
- gli embedding BERT delle descrizioni dei cani;

per generare una classifica dei cani più compatibili con un profilo adottante.

Il sistema distingue tra:

- **vincoli forti**, che possono escludere o portare a punteggio nullo;
- **preferenze morbide**, che contribuiscono al punteggio finale.

## 1. Import delle librerie

Vengono importate le librerie necessarie per caricare i dati, calcolare gli score di compatibilità e confrontare semanticamente i testi tramite BERT.

In [13]:
import pandas as pd
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

## 2. Caricamento dei dati

Vengono caricati:

- il dataset pulito dei cani (`dogs_clean.csv`);
- gli embedding BERT delle descrizioni degli annunci (`bert_embeddings.npy`), calcolati nel notebook precedente.

In [14]:
dogs = pd.read_csv("../data/processed/dogs_clean.csv")
bert_embeddings = np.load("../data/processed/bert_embeddings.npy")

print("Dataset cani:", dogs.shape)
print("Embedding BERT:", bert_embeddings.shape)

dogs.head()

Dataset cani: (8132, 30)
Embedding BERT: (8132, 768)


,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,Color3,MaturitySize,...,Description,PetID,PhotoAmt,AdoptionSpeed,description_len,has_description,age_group,gender_label,fur_length_label,maturity_size_label
0,1,Brisco,1,307,0,1,2,7,0,2,...,Their pregnant mother was dumped by her irresp...,3422e4906,7.0,3,393,1,puppy,male,medium,medium
1,1,Miko,4,307,0,2,1,2,0,2,...,"Good guard dog, very alert, active, obedience ...",5842f1ff5,8.0,2,146,1,puppy,female,short,medium
2,1,Hunter,1,307,0,1,1,0,0,2,...,This handsome yet cute boy is up for adoption....,850a43f90,3.0,2,390,1,puppy,male,short,medium
3,1,Siu Pak & Her 6 Puppies,0,307,0,2,1,2,7,2,...,Siu Pak just give birth on 13/6/10 to 6puppies...,97aa9eeac,9.0,3,109,1,puppy,female,short,medium
4,1,Bear,2,307,0,1,1,2,7,2,...,"For serious adopter, please do sms or call for...",8b693ca84,7.0,1,68,1,puppy,male,short,medium


## 3. Profilo famiglia adottante

In questa sezione viene definito un esempio di profilo famiglia. I campi corrispondono al modulo compilato dall'adottante.

Il testo libero verrà confrontato tramite BERT con la descrizione testuale dei cani.

In [ ]:
family_profile = {
    "applicant_age": "31-60",          # 18-30, 31-60, over_60
    "children": True,
    "dog_experience": "some",          # none, some, multiple, high
    "housing": "house",                # apartment, house, countryside
    "garden": True,
    "preferred_gender": "indifferent", # male, female, indifferent
    "preferred_age": "puppy",          # puppy, young, adult, senior, indifferent
    "preferred_size": "medium",        # small, medium, large, extra_large, indifferent
    "preferred_fur": "short",          # short, medium, long, indifferent
    "daily_time": "2-4",               # <1, 1-2, 2-4, >4
    "activity_level": "moderate",      # calm, moderate, active
    "ideal_dog_description": (
        "We are looking for a friendly and affectionate dog, suitable for a family with children, "
        "not too large, playful but also calm at home."
    )
}

family_profile

{'applicant_age': 'over_60',
 'children': True,
 'dog_experience': 'some',
 'housing': 'house',
 'garden': True,
 'preferred_gender': 'indifferent',
 'preferred_age': 'puppy',
 'preferred_size': 'medium',
 'preferred_fur': 'short',
 'daily_time': '2-4',
 'activity_level': 'moderate',
 'ideal_dog_description': 'We are looking for a friendly and affectionate dog, suitable for a family with children, not too large, playful but also calm at home.'}

## 4. Regole di compatibilità strutturata

La compatibilità strutturata viene calcolata confrontando le preferenze della famiglia con le caratteristiche del cane.

Il sistema usa due tipi di regole:

### Vincoli forti

Queste regole possono assegnare direttamente punteggio 0, perché indicano una situazione poco adatta:

- cane `extra_large` senza giardino;
- cane `extra_large` in appartamento senza giardino;
- richiedente con meno di 1 ora al giorno disponibile.

### Preferenze morbide

Queste regole non escludono automaticamente il cane, ma aumentano o riducono il punteggio:

- sesso preferito;
- fascia d'età preferita;
- taglia preferita;
- lunghezza del pelo preferita;
- presenza di bambini;
- presenza di giardino;
- esperienza con cani;
- età del richiedente.

Il risultato finale viene normalizzato nell'intervallo [0,1].

In [16]:
def match_preference(preference, dog_value):
    """Returns 1 if the preference is indifferent or matches the dog value, otherwise 0."""
    if preference == "indifferent":
        return 1.0
    return 1.0 if preference == dog_value else 0.0


def size_similarity(preferred_size, dog_size):
    """Returns a partial score for dog size compatibility."""
    if preferred_size == "indifferent":
        return 1.0

    size_order = {
        "small": 0,
        "medium": 1,
        "large": 2,
        "extra_large": 3
    }

    if preferred_size not in size_order or dog_size not in size_order:
        return 0.0

    distance = abs(size_order[preferred_size] - size_order[dog_size])

    if distance == 0:
        return 1.0
    elif distance == 1:
        return 0.5
    else:
        return 0.0


def compute_structured_score(dog, family):
    """Computes a structured compatibility score between a dog and a family profile."""

    # HARD CONSTRAINT 1:
    # If the family has less than one hour per day, adoption is not recommended.
    if family["daily_time"] == "<1":
        return 0.0

    # HARD CONSTRAINT 2:
    # Extra large dogs require a garden.
    if dog["maturity_size_label"] == "extra_large" and not family["garden"]:
        return 0.0

    # HARD CONSTRAINT 3:
    # Extra large dogs are excluded for apartments without a garden.
    if (
        dog["maturity_size_label"] == "extra_large"
        and family["housing"] == "apartment"
        and not family["garden"]
    ):
        return 0.0

    scores = []

    # Direct preferences.
    scores.append(match_preference(family["preferred_gender"], dog["gender_label"]))
    scores.append(match_preference(family["preferred_age"], dog["age_group"]))
    scores.append(size_similarity(family["preferred_size"], dog["maturity_size_label"]))
    scores.append(match_preference(family["preferred_fur"], dog["fur_length_label"]))

    # Children rule: if children are present, puppies/young dogs are slightly preferred.
    if family["children"]:
        scores.append(1.0 if dog["age_group"] in ["puppy", "young"] else 0.5)

    # Garden rule:
    # - large dogs without a garden are strongly penalized;
    # - extra large dogs without a garden have already been excluded.
    if dog["maturity_size_label"] == "large":
        scores.append(1.0 if family["garden"] else 0.2)
    elif dog["maturity_size_label"] == "extra_large":
        scores.append(1.0)
    else:
        scores.append(1.0)

    # Time availability rule.
    if family["daily_time"] == "1-2":
        scores.append(0.6 if dog["age_group"] == "puppy" else 0.8)
    else:
        scores.append(1.0)

    # Experience rule:
    # no experience strongly penalizes large dogs and puppies;
    # extra large dogs receive score 0 for this component.
    if family["dog_experience"] == "none":
        if dog["maturity_size_label"] == "extra_large":
            scores.append(0.0)
        elif dog["maturity_size_label"] == "large":
            scores.append(0.2)
        elif dog["age_group"] == "puppy":
            scores.append(0.5)
        else:
            scores.append(1.0)
    else:
        scores.append(1.0)

    # Older applicant rule:
    # senior applicants are oriented toward adult/senior dogs.
    if family["applicant_age"] == "over_60":
        scores.append(1.0 if dog["age_group"] in ["adult", "senior"] else 0.4)

    return np.mean(scores)

## 5. Calcolo della compatibilità strutturata

La funzione viene applicata a tutti i cani del dataset.

In [17]:
dogs["structured_score"] = dogs.apply(
    lambda row: compute_structured_score(row, family_profile),
    axis=1
)

dogs[[
    "Name",
    "age_group",
    "gender_label",
    "maturity_size_label",
    "fur_length_label",
    "structured_score"
]].head()

,Name,age_group,gender_label,maturity_size_label,fur_length_label,structured_score
0,Brisco,puppy,male,medium,medium,0.822222
1,Miko,puppy,female,medium,short,0.933333
2,Hunter,puppy,male,medium,short,0.933333
3,Siu Pak & Her 6 Puppies,puppy,female,medium,short,0.933333
4,Bear,puppy,male,medium,short,0.933333


## 6. Similarità semantica tramite BERT

La descrizione del cane ideale inserita dalla famiglia viene trasformata in embedding BERT.

Successivamente viene calcolata la similarità coseno tra:

- embedding della descrizione della famiglia;
- embedding della descrizione di ogni cane.

Questa misura permette di stimare quanto il testo libero dell'adottante sia semanticamente vicino alla descrizione dell'annuncio.

In [18]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name)
bert_model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

In [19]:
def get_bert_embedding(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = bert_model(**inputs)

    embedding = outputs.last_hidden_state.mean(dim=1)
    return embedding.squeeze().numpy()


family_embedding = get_bert_embedding(family_profile["ideal_dog_description"])
print(family_embedding.shape)

(768,)


In [20]:
semantic_scores = cosine_similarity(
    family_embedding.reshape(1, -1),
    bert_embeddings
).flatten()

# Cosine similarity can theoretically range from -1 to 1.
# Here it is normalized to [0,1] for combination with the structured score.
dogs["bert_similarity"] = (semantic_scores + 1) / 2

dogs[["Name", "Description", "bert_similarity"]].head()

,Name,Description,bert_similarity
0,Brisco,Their pregnant mother was dumped by her irresp...,0.907731
1,Miko,"Good guard dog, very alert, active, obedience ...",0.885574
2,Hunter,This handsome yet cute boy is up for adoption....,0.913940
3,Siu Pak & Her 6 Puppies,Siu Pak just give birth on 13/6/10 to 6puppies...,0.832422
4,Bear,"For serious adopter, please do sms or call for...",0.837586


## 7. Score finale di compatibilità

Lo score finale combina:

- 70% compatibilità strutturata;
- 30% similarità semantica tramite BERT.

La componente strutturata ha peso maggiore perché rappresenta vincoli e preferenze esplicite della famiglia. La componente testuale permette invece di catturare aspetti caratteriali e descrittivi non direttamente rappresentati dalle variabili del dataset.

In [21]:
dogs["final_score"] = (
    0.7 * dogs["structured_score"] +
    0.3 * dogs["bert_similarity"]
)

dogs[["Name", "structured_score", "bert_similarity", "final_score"]].head()

,Name,structured_score,bert_similarity,final_score
0,Brisco,0.822222,0.907731,0.847875
1,Miko,0.933333,0.885574,0.919006
2,Hunter,0.933333,0.913940,0.927515
3,Siu Pak & Her 6 Puppies,0.933333,0.832422,0.903060
4,Bear,0.933333,0.837586,0.904609


## 8. Classifica finale dei cani consigliati

I cani vengono ordinati in base allo score finale. La classifica rappresenta il risultato del prototipo di recommendation system.

In [22]:
ranking = dogs.sort_values(
    by="final_score",
    ascending=False
)[[
    "Name",
    "Age",
    "age_group",
    "gender_label",
    "maturity_size_label",
    "fur_length_label",
    "PhotoAmt",
    "structured_score",
    "bert_similarity",
    "final_score",
    "Description"
]].head(10)

ranking

,Name,Age,age_group,gender_label,maturity_size_label,fur_length_label,PhotoAmt,structured_score,bert_similarity,final_score,Description
2433,Leo,3,puppy,male,medium,short,3.0,0.933333,0.943833,0.936483,Leo is a stray awaiting a nice place to be cal...
5902,Mick,3,puppy,male,medium,short,5.0,0.933333,0.943298,0.936323,Mick is a very handsome mixed breed pup. A bit...
5663,Chino,1,puppy,male,medium,short,11.0,0.933333,0.940831,0.935583,Chino is among 7 siblings that need a good hom...
2119,Gruff,3,puppy,male,medium,short,8.0,0.933333,0.939150,0.935078,Gruff is a handsome looking black dog. He is p...
2999,Zara,2,puppy,female,medium,short,13.0,0.933333,0.938650,0.934928,Zara is a beautiful young doggie with nice bla...
1566,Mr Zorro,6,puppy,male,medium,short,11.0,0.933333,0.938196,0.934792,"Mr Zorro is an ultra friendly, silly, playful ..."
2231,Mina,3,puppy,female,medium,short,5.0,0.933333,0.937850,0.934688,A mixed breed female dog gave birth to a litte...
5799,"Grey-eyed, Brown Male Puppy",2,puppy,male,medium,short,2.0,0.933333,0.937832,0.934683,This cute grey-eyed boy was found in our taman...
6980,Zoey,1,puppy,female,medium,short,11.0,0.933333,0.937322,0.934530,"Zoey has a cream color coat, with light grey h..."
3267,Socks,2,puppy,male,medium,short,3.0,0.933333,0.936690,0.934340,ADOPTION FEE COVERS DESEXING AND VACCINES. Soc...


## 9. Spiegazione della raccomandazione

Per rendere il sistema più interpretabile, viene generata una breve spiegazione automatica per ciascun cane consigliato.

In [23]:
def explain_recommendation(dog, family):
    reasons = []

    if family["preferred_age"] == "indifferent" or dog["age_group"] == family["preferred_age"]:
        reasons.append("fascia d'età compatibile")

    if family["preferred_size"] == "indifferent" or dog["maturity_size_label"] == family["preferred_size"]:
        reasons.append("taglia compatibile")

    if family["preferred_gender"] == "indifferent" or dog["gender_label"] == family["preferred_gender"]:
        reasons.append("sesso compatibile")

    if family["preferred_fur"] == "indifferent" or dog["fur_length_label"] == family["preferred_fur"]:
        reasons.append("lunghezza del pelo compatibile")

    if dog["bert_similarity"] > dogs["bert_similarity"].quantile(0.75):
        reasons.append("descrizione semanticamente vicina al cane ideale")

    if dog["structured_score"] == 0:
        reasons.append("presenza di vincoli non compatibili")

    if len(reasons) == 0:
        return "Compatibilità basata sul punteggio complessivo."

    return ", ".join(reasons)


ranking_with_explanations = ranking.copy()
ranking_with_explanations["explanation"] = ranking_with_explanations.apply(
    lambda row: explain_recommendation(row, family_profile),
    axis=1
)

ranking_with_explanations[["Name", "final_score", "explanation"]]

,Name,final_score,explanation
2433,Leo,0.936483,"fascia d'età compatibile, taglia compatibile, ..."
5902,Mick,0.936323,"fascia d'età compatibile, taglia compatibile, ..."
5663,Chino,0.935583,"fascia d'età compatibile, taglia compatibile, ..."
2119,Gruff,0.935078,"fascia d'età compatibile, taglia compatibile, ..."
2999,Zara,0.934928,"fascia d'età compatibile, taglia compatibile, ..."
1566,Mr Zorro,0.934792,"fascia d'età compatibile, taglia compatibile, ..."
2231,Mina,0.934688,"fascia d'età compatibile, taglia compatibile, ..."
5799,"Grey-eyed, Brown Male Puppy",0.934683,"fascia d'età compatibile, taglia compatibile, ..."
6980,Zoey,0.934530,"fascia d'età compatibile, taglia compatibile, ..."
3267,Socks,0.934340,"fascia d'età compatibile, taglia compatibile, ..."


## 10. Verifica dei vincoli

Questa cella permette di osservare quanti cani ricevono score strutturato nullo a causa dei vincoli forti.

In [24]:
excluded_dogs = dogs[dogs["structured_score"] == 0]

print("Numero di cani esclusi o fortemente non compatibili:", excluded_dogs.shape[0])

excluded_dogs[[
    "Name",
    "age_group",
    "maturity_size_label",
    "structured_score"
]].head()

Numero di cani esclusi o fortemente non compatibili: 0


,Name,age_group,maturity_size_label,structured_score


## 11. Conclusioni

Il prototipo dimostra come i modelli sviluppati nei notebook precedenti possano essere utilizzati per costruire un sistema di raccomandazione cane–famiglia.

Il sistema combina:

- regole basate su preferenze esplicite;
- vincoli forti per evitare abbinamenti poco adatti;
- similarità semantica tramite BERT.

Il risultato è una classifica personalizzata dei cani più compatibili con il profilo dell'adottante.

Questa parte rappresenta un'estensione rispetto alla semplice previsione della velocità di adozione, perché introduce una logica personalizzata orientata al supporto decisionale.